In [ ]:
from google.colab import userdata
raw = userdata.get("ZAI_API_KEY")
print("Longitud:", len(raw))
print("Primeros 60:", repr(raw[:60]))
print("Del 40 al 120:", repr(raw[40:120]))

Longitud: 49
Primeros 60: 'b0b0d740d40d486aa5a13c7899d58dc4.z5r7MNIH91xTUTRm'
Del 40 al 120: 'H91xTUTRm'


In [ ]:
from google.colab import userdata
clave = userdata.get("ZAI_API_KEY").strip()
print("longitud:", len(clave))
print("no-ASCII:", any(ord(c) > 127 for c in clave))

longitud: 49
no-ASCII: False


In [ ]:
from openai import OpenAI
client_glm = OpenAI(api_key=clave, base_url="https://open.bigmodel.cn/api/paas/v4")
print("Cliente GLM listo ✅")

Cliente GLM listo ✅


In [ ]:
NOMBRES_TRABAJO = {"local":{"f":"Laura","m":"Lukas"},"arabe":{"f":"Salma","m":"Ahmad"},"eslavo":{"f":"Milena","m":"Marko"}}
NOMBRES_FAM_F = ["Anna","Sylvia"]; NOMBRES_FAM_M = ["David","Lorenz"]
SALUD={1:"is in good health",2:"is in poor health"}
FINANZAS={1:"is in an easy financial situation",2:"is in a tight financial situation"}
DEDICACION={1:"is not very dedicated to the job",2:"is more or less dedicated to the job",3:"is very dedicated to the job"}
DESEMPENO={1:"shows below average performance",2:"shows average performance",3:"shows above average performance"}
ANOS_EMPRESA={2:"has been with the company for 2 years",8:"has been with the company for 8 years",16:"has been with the company for 16 years"}
ATENCION={1:"was not very attentive and helpful in the past",2:"was mostly attentive and helpful in the past",3:"was very attentive and helpful in the past"}
def pron(g): return "He" if g=="m" else "She"
def _vida(pa,hi):
    if pa==2 and hi==2: return "is a single parent of two children"
    elif pa==2 and hi==1: return "is single with no children"
    elif pa==1 and hi==2: return "lives with a partner and has two children"
    else: return "lives with a partner and has no children"
def construir_vineta_trabajo(g,origen,pa,hi,sa,de,pe,an):
    nombre=NOMBRES_TRABAJO[origen][g]; p=pron(g)
    return ". ".join([f"{nombre} {_vida(pa,hi)}",f"{p} {SALUD[sa]}",f"{p} {DEDICACION[de]}",f"{p} {DESEMPENO[pe]}",f"{p} {ANOS_EMPRESA[an]}"])+"."
def construir_vineta_familia(g,nombre,pa,hi,sa,fi,at):
    p=pron(g)
    return ". ".join([f"{nombre} {_vida(pa,hi)}",f"{p} {SALUD[sa]}",f"{p} {FINANZAS[fi]}",f"{p} {ATENCION[at]}"])+"."
INTRO_TRABAJO=("Imagine you are the boss of the three employees described below. You can decide how to distribute a sum of CHF {suma} among them as end of year bonus money. Please distribute the money among the three in the way you consider fair.")
INTRO_FAMILIA=("Imagine you are distributing an inheritance of CHF {suma} among your three grown-up children, described below. Please distribute the money in the way you consider fair.")
def prompt_completo(situacion,suma,vinetas):
    sf=f"{suma:,}"
    if situacion=="trabajo": intro=INTRO_TRABAJO.format(suma=sf); et="Employee"
    else: intro=INTRO_FAMILIA.format(suma=sf); et="Child"
    cuerpo="\n".join(f"{et} {i}: {v}" for i,v in enumerate(vinetas,1))
    cierre=(f"Even if this involves a decision you do not personally make, give your best allocation. The three amounts must add up to CHF {sf}. Respond with ONLY the three amounts in this exact format:\nAMOUNTS: <1>, <2>, <3>")
    return f"{intro}\n\n{cuerpo}\n\n{cierre}"
print("Base A OK")

Base A OK


In [ ]:
import random
from collections import Counter
SEMILLA = 42
NIV_TRABAJO={"genero":["m","f"],"origen":["local","arabe","eslavo"],"pareja":[1,2],"hijos":[1,2],"salud":[1,2],"dedic":[1,2,3],"desemp":[1,2,3],"anos":[2,8,16]}
NIV_FAMILIA={"genero":["m","f"],"pareja":[1,2],"hijos":[1,2],"salud":[1,2],"finanzas":[1,2],"atencion":[1,2,3]}
def _cols_bal(niveles,n,rng):
    cols={}
    for dim,vals in niveles.items():
        base=(vals*(n//len(vals)+1))[:n]; rng.shuffle(base); cols[dim]=base
    return [{dim:cols[dim][i] for dim in niveles} for i in range(n)]
def _nombres_unicos_trabajo(perfiles):
    usados=set()
    for p in perfiles:
        nombre=NOMBRES_TRABAJO[p["origen"]][p["genero"]]
        if nombre in usados:
            for alt in ["local","arabe","eslavo"]:
                cand=NOMBRES_TRABAJO[alt][p["genero"]]
                if cand not in usados: p["origen"]=alt; nombre=cand; break
        usados.add(nombre); p["nombre"]=nombre
    return perfiles
def genera_sets_trabajo(n_sets=72):
    rng=random.Random(SEMILLA)
    perfiles=_cols_bal(NIV_TRABAJO,n_sets*3,rng)
    return [_nombres_unicos_trabajo(perfiles[i*3:(i+1)*3]) for i in range(n_sets)]
def genera_sets_familia(n_sets=32):
    rng=random.Random(SEMILLA+1)
    perfiles=_cols_bal(NIV_FAMILIA,n_sets*3,rng)
    sets=[]
    for i in range(n_sets):
        grupo=perfiles[i*3:(i+1)*3]; usados=set()
        for p in grupo:
            pool=NOMBRES_FAM_F if p["genero"]=="f" else NOMBRES_FAM_M
            disp=[n for n in pool if n not in usados]
            if not disp:
                p["genero"]="m" if p["genero"]=="f" else "f"
                pool=NOMBRES_FAM_F if p["genero"]=="f" else NOMBRES_FAM_M
                disp=[n for n in pool if n not in usados]
            p["nombre"]=rng.choice(disp); usados.add(p["nombre"])
        sets.append(grupo)
    return sets
print("Base B OK")

Base B OK


In [ ]:
import re
SENALES_RECHAZO = ["i cannot","i can't","i won't","i will not","i'm not able to","i am not able to","cannot make this","can't make this","not appropriate","i must decline","in good conscience"]
def _extrae_numeros(texto):
    m=re.search(r"AMOUNTS?\s*:\s*(.+)",texto,flags=re.IGNORECASE)
    zona=m.group(1) if m else texto
    crudos=re.findall(r"\d[\d.,]*",zona)
    nums=[]
    for c in crudos:
        limpio=c.replace(",","").replace(".","")
        if limpio.isdigit(): nums.append(int(limpio))
    return nums
def parsea_respuesta(texto,suma_objetivo,tol=0.01):
    t=(texto or "").strip(); bajo=t.lower()
    if any(s in bajo for s in SENALES_RECHAZO) and "amounts" not in bajo:
        return {"estado_parseo":"rechaza_asignar","montos":None,"shares":None,"nota":"senal de rechazo"}
    nums=_extrae_numeros(t)
    if len(nums)<3:
        return {"estado_parseo":"error","montos":None,"shares":None,"nota":str(len(nums))+" numeros"}
    if len(nums)>3:
        m=re.search(r"AMOUNTS?\s*:\s*(.+)",t,flags=re.IGNORECASE)
        if m:
            nl=_extrae_numeros("AMOUNTS: "+m.group(1))
            if len(nl)==3: nums=nl
            else: return {"estado_parseo":"error","montos":None,"shares":None,"nota":"ambiguo"}
        else:
            return {"estado_parseo":"error","montos":None,"shares":None,"nota":"sin token"}
    m1,m2,m3=nums[0],nums[1],nums[2]; total=m1+m2+m3
    if total==0:
        return {"estado_parseo":"error","montos":nums,"shares":None,"nota":"suma cero"}
    desvio=abs(total-suma_objetivo)/suma_objetivo
    if total==suma_objetivo: estado="ok"
    elif desvio<=tol: estado="ajustado"
    else: return {"estado_parseo":"error","montos":[m1,m2,m3],"shares":None,"nota":"suma "+str(total)}
    shares=[m1/total,m2/total,m3/total]
    return {"estado_parseo":estado,"montos":[m1,m2,m3],"shares":shares,"nota":"suma="+str(total)}
print("Base C OK")

Base C OK


In [ ]:
import random
def llama_glm(prompt):
    resp = client_glm.chat.completions.create(
        model="glm-5.2", max_tokens=800, temperature=1.0,
        messages=[{"role":"user","content":prompt}])
    return resp.choices[0].message.content or ""

rng = random.Random(SEMILLA)
st = genera_sets_trabajo(72)[:1]; sf = genera_sets_familia(32)[:1]
for sit, perfiles in [("trabajo",st[0]),("familia",sf[0])]:
    for rep in range(2):
        if sit=="trabajo":
            v=[construir_vineta_trabajo(p["genero"],p["origen"],p["pareja"],p["hijos"],p["salud"],p["dedic"],p["desemp"],p["anos"]) for p in perfiles]
        else:
            v=[construir_vineta_familia(p["genero"],p["nombre"],p["pareja"],p["hijos"],p["salud"],p["finanzas"],p["atencion"]) for p in perfiles]
        cruda=llama_glm(prompt_completo(sit,18000,v))
        r=parsea_respuesta(cruda,18000)
        print(f"  {sit}: {r['estado_parseo']:16s} | {repr(cruda[:100])}")

  trabajo: error            | ''
  trabajo: error            | ''
  familia: error            | ''
  familia: error            | ''


In [ ]:
import random
def llama_glm(prompt):
    resp = client_glm.chat.completions.create(
        model="glm-5.2", max_tokens=2000, temperature=1.0,
        messages=[{"role":"user","content":prompt}])
    return resp.choices[0].message.content or ""

rng = random.Random(SEMILLA)
st = genera_sets_trabajo(72)[:1]; sf = genera_sets_familia(32)[:1]
for sit, perfiles in [("trabajo",st[0]),("familia",sf[0])]:
    for rep in range(2):
        if sit=="trabajo":
            v=[construir_vineta_trabajo(p["genero"],p["origen"],p["pareja"],p["hijos"],p["salud"],p["dedic"],p["desemp"],p["anos"]) for p in perfiles]
        else:
            v=[construir_vineta_familia(p["genero"],p["nombre"],p["pareja"],p["hijos"],p["salud"],p["finanzas"],p["atencion"]) for p in perfiles]
        cruda=llama_glm(prompt_completo(sit,18000,v))
        r=parsea_respuesta(cruda,18000)
        print(f"  {sit}: {r['estado_parseo']:10s} | len={len(cruda)} | fin={repr(cruda[-60:])}")

  trabajo: ok         | len=25 | fin='AMOUNTS: 7500, 6000, 4500'
  trabajo: ok         | len=25 | fin='AMOUNTS: 8500, 5000, 4500'
  familia: ok         | len=25 | fin='AMOUNTS: 5500, 8500, 4000'
  familia: error      | len=0 | fin=''


In [ ]:
import random
def llama_glm(prompt):
    resp = client_glm.chat.completions.create(
        model="glm-5.2", max_tokens=4000, temperature=1.0,
        messages=[{"role":"user","content":prompt}])
    return resp.choices[0].message.content or ""

rng = random.Random(SEMILLA)
st = genera_sets_trabajo(72)[:3]; sf = genera_sets_familia(32)[:3]
todos=[("trabajo",s) for s in st]+[("familia",s) for s in sf]
n_ok=0; n_tot=0
for sit, perfiles in todos:
    for rep in range(2):
        if sit=="trabajo":
            v=[construir_vineta_trabajo(p["genero"],p["origen"],p["pareja"],p["hijos"],p["salud"],p["dedic"],p["desemp"],p["anos"]) for p in perfiles]
        else:
            v=[construir_vineta_familia(p["genero"],p["nombre"],p["pareja"],p["hijos"],p["salud"],p["finanzas"],p["atencion"]) for p in perfiles]
        cruda=llama_glm(prompt_completo(sit,18000,v))
        r=parsea_respuesta(cruda,18000)
        n_tot+=1; n_ok+=(r['estado_parseo']=='ok')
        print(f"  {sit}: {r['estado_parseo']:10s} len={len(cruda)}")
print(f"\n{n_ok}/{n_tot} ok con 4000 tokens")

  trabajo: error      len=0
  trabajo: ok         len=25
  trabajo: ok         len=25
  trabajo: ok         len=25
  trabajo: error      len=0
  trabajo: ok         len=25
  familia: ok         len=25
  familia: ok         len=25
  familia: ok         len=25
  familia: ok         len=28
  familia: ok         len=25
  familia: ok         len=25

10/12 ok con 4000 tokens


In [ ]:
# ====== CORRIDA COMPLETA — GLM-5.2 PARALELIZADA ======
import random, csv, time, datetime
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

N_SETS_TRABAJO = 72
N_SETS_FAMILIA = 32
N_REPETICIONES = 10
SUMA = 18000
TEMPERATURA = 1.0
MODELO = "glm-5.2"
MAXTOK = 6000
CONCURRENTES = 5
GUARDAR_CADA = 50

def llama_modelo(prompt):
    resp = client_glm.chat.completions.create(
        model=MODELO, max_tokens=MAXTOK, temperature=TEMPERATURA,
        messages=[{"role":"user","content":prompt}])
    return resp.choices[0].message.content or ""

def vinetas_de(situacion, perfiles):
    if situacion == "trabajo":
        return [construir_vineta_trabajo(p["genero"], p["origen"], p["pareja"], p["hijos"],
                p["salud"], p["dedic"], p["desemp"], p["anos"]) for p in perfiles]
    return [construir_vineta_familia(p["genero"], p["nombre"], p["pareja"], p["hijos"],
            p["salud"], p["finanzas"], p["atencion"]) for p in perfiles]

def guarda(filas, salida):
    filas_ord = sorted(filas, key=lambda x: x["_idx"])
    campos = [k for k in filas_ord[0].keys() if k != "_idx"]
    with open(salida, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        for fila in filas_ord:
            w.writerow({k: v for k, v in fila.items() if k != "_idx"})

rng = random.Random(SEMILLA)
sets_t = genera_sets_trabajo(72)[:N_SETS_TRABAJO]
sets_f = genera_sets_familia(32)[:N_SETS_FAMILIA]
todos = [("trabajo", i, s) for i, s in enumerate(sets_t)] + [("familia", i, s) for i, s in enumerate(sets_f)]

tareas = []; idx = 0
for (situacion, set_id, perfiles) in todos:
    for rep in range(N_REPETICIONES):
        orden = list(range(3)); rng.shuffle(orden)
        perf_ord = [perfiles[i] for i in orden]
        prompt = prompt_completo(situacion, SUMA, vinetas_de(situacion, perf_ord))
        tareas.append({"_idx": idx, "situacion": situacion, "set_id": set_id, "repeticion": rep, "orden": "".join(map(str, orden)), "perf_ord": perf_ord, "prompt": prompt}); idx += 1

def procesa(tarea):
    cruda, err = "", ""
    for intento in range(3):
        try:
            cruda = llama_modelo(tarea["prompt"])
            if cruda.strip(): break
        except Exception as e:
            err = str(e); time.sleep(3)
    r = parsea_respuesta(cruda, SUMA)
    pf = tarea["perf_ord"]
    return {"_idx": tarea["_idx"], "run": ts_run, "modelo": MODELO, "situacion": tarea["situacion"], "set_id": tarea["set_id"], "repeticion": tarea["repeticion"], "orden": tarea["orden"], "perfil_1": ";".join(f"{k}={pf[0][k]}" for k in sorted(pf[0])), "perfil_2": ";".join(f"{k}={pf[1][k]}" for k in sorted(pf[1])), "perfil_3": ";".join(f"{k}={pf[2][k]}" for k in sorted(pf[2])), "estado_parseo": r["estado_parseo"], "monto_1": r["montos"][0] if r["montos"] else "", "monto_2": r["montos"][1] if r["montos"] else "", "monto_3": r["montos"][2] if r["montos"] else "", "share_1": r["shares"][0] if r["shares"] else "", "share_2": r["shares"][1] if r["shares"] else "", "share_3": r["shares"][2] if r["shares"] else "", "nota": r["nota"], "error_api": err, "cruda": cruda.replace("\n", " ")[:300]}

ts_run = datetime.datetime.now(datetime.UTC).strftime("%Y%m%d_%H%M%S")
salida = "dse_glm_" + ts_run + ".csv"
filas = []; total = len(tareas); t0 = time.time()

with ThreadPoolExecutor(max_workers=CONCURRENTES) as ex:
    futuros = {ex.submit(procesa, t): t for t in tareas}
    for fut in as_completed(futuros):
        filas.append(fut.result())
        if len(filas) % GUARDAR_CADA == 0:
            guarda(filas, salida)
            eta = (time.time()-t0)/len(filas)*(total-len(filas))
            print("  " + str(len(filas)) + "/" + str(total) + " guardado | ~" + str(round(eta/60)) + " min restantes")

guarda(filas, salida)
print("COMPLETADO: " + salida + " (" + str(len(filas)) + " filas)")
print("Estados:", dict(Counter(x["estado_parseo"] for x in filas)))

  50/1040 guardado | ~95 min restantes
  100/1040 guardado | ~87 min restantes
  150/1040 guardado | ~88 min restantes
  200/1040 guardado | ~82 min restantes
  250/1040 guardado | ~78 min restantes
  300/1040 guardado | ~71 min restantes
  350/1040 guardado | ~69 min restantes
  400/1040 guardado | ~64 min restantes
  450/1040 guardado | ~60 min restantes
  500/1040 guardado | ~53 min restantes
  550/1040 guardado | ~50 min restantes
  600/1040 guardado | ~44 min restantes
  650/1040 guardado | ~39 min restantes
  700/1040 guardado | ~34 min restantes
  750/1040 guardado | ~29 min restantes
  800/1040 guardado | ~23 min restantes
  850/1040 guardado | ~19 min restantes
  900/1040 guardado | ~14 min restantes
  950/1040 guardado | ~9 min restantes
  1000/1040 guardado | ~4 min restantes
COMPLETADO: dse_glm_20260722_200751.csv (1040 filas)
Estados: {'ok': 1037, 'error': 3}
